In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. 数据生成
np.random.seed(42)
n = 1000
data = pd.DataFrame({
    "age": np.random.randint(18, 65, n),
    "gender": np.random.choice(["男", "女"], n),
    "browse_duration": np.round(np.random.exponential(30, n), 1),
    "add_to_cart": np.random.poisson(3, n),
    "history_purchases": np.random.randint(0, 50, n),
})
score = (0.03 * data["age"] + 0.5 * (data["gender"] == "女").astype(int)
         + 0.05 * data["browse_duration"] + 0.3 * data["add_to_cart"]
         + 0.1 * data["history_purchases"] - 4)
data["purchased"] = (np.random.random(n) < (1 / (1 + np.exp(-score)))).astype(int)

# 2. 特征工程
data["gender_encoded"] = LabelEncoder().fit_transform(data["gender"])
feature_cols = ["age", "gender_encoded", "browse_duration", "add_to_cart", "history_purchases"]
scaler = StandardScaler()
X = scaler.fit_transform(data[feature_cols].values)
y = data["purchased"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. 建模调优
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

# 4. 最终评估
y_pred = best_model.predict(X_test)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=['未购买', '已购买'])}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\n最佳参数: {grid_search.best_params_}")

Accuracy:  0.8250

Classification Report:
              precision    recall  f1-score   support

         未购买       0.60      0.31      0.41        39
         已购买       0.85      0.95      0.90       161

    accuracy                           0.82       200
   macro avg       0.72      0.63      0.65       200
weighted avg       0.80      0.82      0.80       200

Confusion Matrix:
[[ 12  27]
 [  8 153]]

最佳参数: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}


In [2]:
joblib.dump(best_model, "purchase_predict_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("模型和缩放器已保存")

模型和缩放器已保存


In [3]:
loaded_model = joblib.load("purchase_predict_model.pkl")
sample = X_test[:1]

In [4]:
print(f"加载模型预测: {loaded_model.predict(sample)}")

加载模型预测: [1]
